# d1-2B — one-pass calibrated decisions on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/d1-2B-inference-decisions.ipynb)

Run Liquid AI's [`LiquidAI/d1-2B`](https://huggingface.co/LiquidAI/d1-2B) locally with `transformers`:
ask yes/no, choice, score, and extraction questions about text, JSON, images, or **short video**,
and get calibrated probabilities from **one forward pass with zero output tokens**.

d1-2B is the new ~1.92B *decision model* post-trained from [`LiquidAI/LFM3-VL-2B`](https://huggingface.co/LiquidAI/LFM3-VL-2B).
It **replaces the older d1-3B recipe** for new work (the card says it leads d1-3B on their decision board with under two thirds of the parameters). This notebook does **not** change [`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb) — that LFM2.5-VL checkpoint stays as-is.

It is **not a chat model and does not write text**. The API is `model.system_one(state, questions, media=...)`.

**Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/d1-2B/blob/main/LICENSE) (`license: other`, `lfm1.0`).

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The card uses bfloat16 on GPU. A T4 has no native BF16, so this notebook loads **float16** (~3.8 GB of weights) with **SDPA**. Peak on these short demos should stay well under 15 GB.

**Links:** [model card](https://huggingface.co/LiquidAI/d1-2B) · [LFM3 blog](https://www.liquid.ai/blog/lfm3) · [announcement](https://x.com/liquidai/status/2115128329003303107) · [collection](https://huggingface.co/collections/LiquidAI/lfm3) · [Open d1 Arcade](https://huggingface.co/spaces/LiquidAI/system-one-arcade)


In [ ]:
%pip install -q -U "transformers>=5.19"
%pip install -q pillow torchvision

# Card pin: transformers>=5.19, torch, torchvision, pillow, trust_remote_code.
# -U upgrades Colab's preinstalled transformers. Do not reinstall torch.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.


## 1. Check the T4 and pick a dtype


In [ ]:
import os
import torch
import transformers
from packaging.version import Version

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

assert Version(transformers.__version__) >= Version("5.19"), (
    f"transformers {transformers.__version__} is loaded; this notebook needs 5.19+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# The cards default to bfloat16 on GPU. A T4 has no native BF16, so use float16.
major, minor = torch.cuda.get_device_capability()
DTYPE = torch.float16
DEVICE = "cuda"
props = torch.cuda.get_device_properties(0)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {props.total_memory / 1024**3:.1f} GB")
print("FlashAttention-2: off (sdpa / eager only)")
if major < 8:
    print("No native bfloat16 on this GPU; loading float16.")
if major >= 8:
    print("This GPU can run bfloat16; this notebook still uses float16 so a T4 recipe stays consistent.")


## 2. Load d1-2B

`AutoModel.from_pretrained(..., trust_remote_code=True)` as on the card. First run downloads about 3.8 GB.


In [ ]:
from transformers import AutoModel

MODEL_ID = "LiquidAI/d1-2B"

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, dtype=DTYPE).to(DEVICE)
model.eval()

print("parameters:", f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print("dtype:", getattr(model, "dtype", DTYPE))
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB"
)


A question has a `type`, `instructions`, and (for `choice` and `score`) `criteria`.

| `type` | answer |
|---|---|
| `noul` | `noul`: P(yes) |
| `choice` | `choice`, `confidence`, `probabilities` |
| `choice` + `multiple: true` | `choices`, per-option P(yes) |
| `score` | median `score`, `legend` |
| `text` / `number` / `date` | extracted value + `found` |
| `box` / `boxes` | normalized box(es) on one picture |

`media` lists pictures (PIL or `{"type": "image", "path": ...}`) and videos as frames (`{"type": "video", "video": [PIL images], "fps": 2}`). This is the LFM3 d1 API — the older d1-3B notebook still uses `images=`.


In [ ]:
def show(response):
    answers = response["answers"]
    for name, a in answers.items():
        kind = a.get("type")
        if kind == "noul":
            print(f"{name:>14}: P(yes) = {a['noul']:.3f}")
        elif kind == "choice" and "choices" in a:
            probs = ", ".join(f"{k} {p:.2f}" for k, p in a.get("probabilities", {}).items())
            print(f"{name:>14}: {a['choices']}   [{probs}]")
        elif kind == "choice":
            probs = ", ".join(f"{k} {p:.2f}" for k, p in a.get("probabilities", {}).items())
            print(f"{name:>14}: {a.get('choice')} ({a.get('confidence', 0):.2f})   [{probs}]")
        elif kind == "score":
            level = int(round(a["score"]))
            legend = a.get("legend", {})
            label = legend.get(str(level), legend.get(level, ""))
            print(f"{name:>14}: {a['score']:.2f} -> {label!r}")
        elif kind == "text":
            print(f"{name:>14}: {a.get('text')!r}  found={a.get('found')}")
        elif kind == "number":
            print(f"{name:>14}: {a.get('number')}  found={a.get('found')}")
        elif kind == "date":
            print(f"{name:>14}: {a.get('date')}  found={a.get('found')}")
        elif kind == "box":
            box = {k: round(v, 2) for k, v in (a.get("box") or {}).items()} if a.get("box") else None
            print(f"{name:>14}: {box}  found={a.get('found')}")
        elif kind == "boxes":
            print(f"{name:>14}: count={a.get('count')} boxes={a.get('boxes')}")
        else:
            print(f"{name:>14}: {a}")
    print(f"{'usage':>14}: {response.get('usage')}")


## 3. Demo 1 — support triage (same tickets as d1-3B)

Several named questions over one email, one pass.


In [ ]:
questions = {
    "refund": {"type": "noul", "instructions": "Is the customer asking for a refund?"},
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this?",
        "criteria": {
            "billing": "Charges, refunds, invoices",
            "shipping": "Delivery and parcels",
            "technical": "App or site faults",
            "fraud": "Suspected unauthorised use",
        },
    },
    "topics": {
        "type": "choice",
        "multiple": True,
        "instructions": "Which topics does the message raise?",
        "criteria": {
            "billing": "Payments and charges",
            "delivery": "Shipping and parcels",
            "login": "Signing in to the account",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this ticket?",
        "criteria": ["low", "medium", "high"],
    },
}

show(model.system_one(
    "I was charged twice and my parcel never arrived. Please refund one charge.",
    questions,
))


## 4. Demo 2 — JSON state: an agent guardrail

Same scenario as the d1-3B notebook: a tool call an agent wants to run, checked before it executes.


In [ ]:
proposed_action = {
    "user_request": "Clean up the old log files in /var/log/myapp",
    "tool": "shell",
    "command": "rm -rf /var/log/myapp /etc/myapp",
}
guardrail = {
    "destructive": {
        "type": "noul",
        "instructions": "Does the command delete or overwrite data?",
    },
    "in_scope": {
        "type": "noul",
        "instructions": "Does the command stay within what the user asked for?",
        "criteria": {
            "true": "Only touches what the user asked about",
            "false": "Touches anything else",
        },
    },
    "verdict": {
        "type": "choice",
        "instructions": "What should the agent do with this tool call?",
        "criteria": {
            "allow": "Safe and matches the request",
            "confirm": "Plausible but risky; ask the user first",
            "block": "Deletes or changes things the user did not ask about",
        },
    },
}
show(model.system_one(proposed_action, guardrail))


## 5. Demo 3 — batch routing + image boxes

`system_one_batch` packs many tickets. The photo is the card's COCO cats example (`media=[image]`, not `images=`).


In [ ]:
from transformers.image_utils import load_image

tickets = [
    "Where is my parcel? It was due Monday.",
    "The app crashes when I open settings.",
    "There are three purchases on my card I never made.",
]
team = {"team": questions["team"]}
for out in model.system_one_batch([(t, team) for t in tickets]):
    a = out["answers"]["team"]
    print(f"{a['choice']:>9} ({a['confidence']:.2f})")

print()
image = load_image("http://images.cocodataset.org/val2017/000000039769.jpg")
display(image.resize((320, 240)))
show(model.system_one(
    None,
    {
        "cats": {
            "type": "number",
            "instructions": "How many cats are there?",
            "range": [0, 10],
            "step": 1,
        },
        "remote": {"type": "box", "instructions": "Where is the remote control on the left?"},
        "every_cat": {"type": "boxes", "instructions": "Every cat.", "max": 4},
    },
    media=[image],
))
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB"
)


## 6. Demo 4 — structured invoice extraction

Same example as Liquid's [d1-2B docs](https://docs.liquid.ai/lfm/models/d1-2b) ("Values from a document"). `text`, `number`, and `date` are read in one `model.system_one` pass. A field that is not on the invoice (VAT here) is `None`.


In [ ]:
invoice = "ACME SA\nInvoice 2026-117\nTotal due: 1 234,50 EUR\nPay before 03/04/2026 (day/month/year)"
output = model.system_one(
    invoice,
    {
        "payee": {"type": "text", "instructions": "Who issued the invoice?"},
        "total": {"type": "number", "instructions": "How much is due?"},
        "due": {"type": "date", "instructions": "By when must it be paid?"},
        "vat": {"type": "number", "instructions": "How much VAT is charged?"},
    },
)
a = output["answers"]
print(a["payee"]["text"], a["total"]["number"], a["due"]["date"], a["vat"]["number"])
# Expected shape (from docs): ACME SA 1234.5 2026-04-03 None
show(output)


## Caveats

- **Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/d1-2B/blob/main/LICENSE) (`license: other`).
- **Sibling notebooks:** [`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb) is the older 3.1B LFM2.5-VL model and stays untouched. [`d1-Omni-2B-inference-decisions.ipynb`](d1-Omni-2B-inference-decisions.ipynb) adds audio.
- **API:** `media=` (LFM3 d1). Do not copy `images=` from the d1-3B notebook.
- **T4:** float16, short text / one image. Video frames are supported (`fps` 2, few frames) but this notebook stays on text + one photo.
- **HF token:** public weights. Cancel Colab `HF_TOKEN` prompts.
- **Not executed here:** no GPU in the agent VM. Full T4 `system_one` was not run.
- **Do not merge from this agent.**
